[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_02_data_structures_algorithms/ch11_algorithms_complexity/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 11 Walkthrough: Algorithmic Thinking, Complexity, Searching, Sorting & Recursion

**Book Reference:** Chapter 11 (Part II Capstone), Sections 11.1 – 11.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections.

## Section 11.3.1 — Complexity Intuition: Empirical $O(n)$ List Scan vs. $O(1)$ Set Lookup

In [ ]:
import time

registry_list = [f"P-{i:06d}" for i in range(40000)]
registry_set = set(registry_list)
probes = [f"P-{i:06d}" for i in range(39000, 40000)]

t0 = time.perf_counter()
hits_list = sum(1 for pid in probes if pid in registry_list)
list_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
hits_set = sum(1 for pid in probes if pid in registry_set)
set_ms = (time.perf_counter() - t0) * 1000

print(f"List scan O(n):  {hits_list} hits in {list_ms:6.2f} ms")
print(f"Set lookup O(1): {hits_set} hits in {set_ms:6.2f} ms")
print(f"Speedup factor:  {list_ms / max(set_ms, 1e-6):.0f}x faster")


## Section 11.3.2 — Binary Search Step-by-Step Halving Trace ($O(\log n)$)

In [ ]:
def trace_binary_search(sorted_ids, target_id):
    """Iterative O(log n) binary search with step-by-step trace."""
    low = 0
    high = len(sorted_ids) - 1
    step = 0
    while low <= high:
        step += 1
        mid = (low + high) // 2
        val = sorted_ids[mid]
        print(f"Step {step}: low={low}, mid={mid}, high={high} -> {val}")
        if val == target_id:
            return mid
        elif val < target_id:
            low = mid + 1
        else:
            high = mid - 1
    return -1

ward_ids = [
    "P-010", "P-020", "P-030", "P-040",
    "P-050", "P-060", "P-070", "P-080",
]
pos = trace_binary_search(ward_ids, "P-070")
print("Found P-070 at index:", pos)


## Section 11.3.3 — Multi-Key Custom Sorting with `sorted(..., key=...)`

In [ ]:
URGENCY_RANK = {"Urgent": 0, "Moderate": 1, "Routine": 2}

triage_raw = [
    {"id": "P-01", "urgency": "Routine", "temp_c": 37.2},
    {"id": "P-02", "urgency": "Urgent", "temp_c": 39.1},
    {"id": "P-03", "urgency": "Moderate", "temp_c": 38.6},
    {"id": "P-04", "urgency": "Urgent", "temp_c": 40.2},
]

ordered = sorted(
    triage_raw,
    key=lambda p: (URGENCY_RANK[p["urgency"]], -p["temp_c"]),
)

print("Sorted triage order:", [p["id"] for p in ordered])
print("Original untouched: ", [p["id"] for p in triage_raw])


## Section 11.3.4 — Recursion & Call-Stack Execution Trace

In [ ]:
def trace_outbreak(initial_cases, r_factor, generations, depth=0):
    """Recursively project outbreak cases while printing call frames."""
    indent = "  " * depth
    print(f"{indent}-> call(gen={generations})")
    if generations == 0:
        base = round(initial_cases, 2)
        print(f"{indent}<- base return {base}")
        return base
    prev = trace_outbreak(initial_cases, r_factor, generations - 1,
                          depth + 1)
    result = round(prev * r_factor, 2)
    print(f"{indent}<- return {prev} * {r_factor} = {result}")
    return result

projected = trace_outbreak(10, 1.5, 3)
print("Projected Gen-3 cases:", projected)


## Section 11.3.5 — Break It On Purpose! (Algorithm & Recursion Error Anatomy)

In [ ]:
# Intentional Bug #1: RecursionError — missing base case
def runaway_outbreak(cases, r_factor, generations):
    return r_factor * runaway_outbreak(cases, r_factor, generations - 1)

runaway_outbreak(10, 1.5, 3)


In [ ]:
# Intentional Bug #2: TypeError — sorting dicts without a key function
sorted(triage_raw)


In [ ]:
# Intentional Bug #3: KeyError — unguarded leaf lookup in contact tree
contact_tree = {"P-01": ["P-02", "P-03"], "P-02": ["P-04"]}

def unsafe_count(tree, node):
    return 1 + sum(unsafe_count(tree, c) for c in tree[node])

unsafe_count(contact_tree, "P-01")


## Section 11.4 — Worked Problem-Structuring Blueprint: End-to-End Outbreak Response Pipeline

In [ ]:
def count_cluster(contact_tree, root_id):
    """Recursively count total cases in a cluster including root_id."""
    total = 1
    for contact in contact_tree.get(root_id, []):
        total += count_cluster(contact_tree, contact)
    return total

def prioritize_clusters(index_patients, ward_table, contact_tree):
    """Join wards O(n+m), count clusters recursively, rank O(k log k)."""
    ward_map = {row["patient_id"]: row["ward"] for row in ward_table}
    summaries = []
    for pid in index_patients:
        size = count_cluster(contact_tree, pid)
        ward = ward_map.get(pid, "Unknown")
        summaries.append((pid, ward, size))
    return sorted(summaries, key=lambda item: (-item[2], item[0]))

indexes = ["P-01", "P-10", "P-20"]
wards = [
    {"patient_id": "P-10", "ward": "Ward B"},
    {"patient_id": "P-01", "ward": "Ward A"},
]
contacts = {
    "P-01": ["P-02", "P-03"],
    "P-02": ["P-04"],
    "P-10": ["P-11", "P-12", "P-13"],
    "P-20": ["P-21"],
}
ranked_clusters = prioritize_clusters(indexes, wards, contacts)
for pid, ward, size in ranked_clusters:
    print(f"{pid} ({ward}): {size} cases in transmission cluster")
